# 06 - Analisis biaya retrieval dan ekspor hasil

Tiga hal: mengukur biaya indeks FAISS yang menjadi satu-satunya biaya tambahan
RM-c, menggabungkan riwayat run bila ada lebih dari satu folder kampanye, dan
mengekspor seluruh artefak ke satu berkas Excel untuk penulisan Bab 4.

In [ ]:
import pandas as pd

from src.config import settings
from src.services.campaign import CampaignRunner

OUT_DIR = settings.default_out_dir
runner = CampaignRunner(out_dir=OUT_DIR)
features = runner.features
print("fitur beku:", {k: v.shape for k, v in features.embeddings.items()})

## 1. Biaya indeks FAISS

RM-c mengklaim nol waktu latih. Klaim itu baru jujur bila biaya retrieval ikut
diukur, dan biayanya terbagi dua: pembangunan indeks sekali dari embedding train,
serta penelusuran pada setiap inferensi yang tumbuh mengikuti k.

In [ ]:
from src.services.faiss_benchmark import FaissBenchmark

benchmark = FaissBenchmark(
    features.embeddings["train"], features.labels["train"], repeats=5
)
hasil = benchmark.run(
    features.embeddings["test"],
    k_values=(1, 3, 5, 10, 20, 50),
    out_dir=settings.output_dir / "faiss_index",
)

print(pd.Series(hasil.build).to_string())
pd.DataFrame(hasil.search)

Indeks bertipe flat/exact, jadi penelusuran adalah brute force atas seluruh
vektor train. Untuk ukuran data ini biayanya sepele dan hasilnya deterministik,
yang jauh lebih penting untuk penelitian daripada penghematan waktu dari indeks
aproksimasi.

In [ ]:
import matplotlib.pyplot as plt

search = pd.DataFrame(hasil.search)
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(search["k"], search["search_time_us_per_query"], marker="o")
ax.set_xlabel("k (jumlah tetangga)")
ax.set_ylabel("mikrodetik per query")
ax.set_title("Biaya penelusuran indeks FAISS")
ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 2. Gabungkan riwayat lintas folder kampanye

Hanya perlu bila ada lebih dari satu folder keluaran, misalnya kampanye utama
ditambah eksplorasi dengan encoder berbeda.

In [ ]:
from src.services.aggregation import RunMerger

SUMBER = {"tuning": OUT_DIR}

if len(SUMBER) > 1:
    ditulis = RunMerger(SUMBER).merge_all(settings.output_dir / "combined")
    for kunci, path in ditulis.items():
        print(f"  {kunci}: {path}")
else:
    print("hanya satu folder kampanye; penggabungan dilewati")

Saat membaca hasil gabungan, kunci barisnya adalah pasangan (`source`, `run_id`)
karena tiap folder memulai penomoran dari 1. Kolom waktu, memori, dan latency
tidak boleh dibandingkan lintas `source`.

## 3. Ekspor ke Excel

In [ ]:
from src.services.workbook import WorkbookBuilder

builder = WorkbookBuilder(OUT_DIR)
path = builder.build(settings.data_dir.parent / "HASIL.xlsx")
print(f"{path} ({path.stat().st_size / 1024:.0f} KB)")

for nama, frame in builder.sheets().items():
    print(f"  {nama:28s}: {len(frame):4,} baris x {len(frame.columns)} kolom")

## 4. Regenerasi seluruh figur

In [ ]:
from src.services.reporting import FigureReporter

reporter = FigureReporter(OUT_DIR)
for skenario in ("rma", "rmb", "rmc"):
    dibuat = reporter.refresh_scenario(skenario)
    print(f"{skenario}: {len(dibuat)} artefak")

reporter.final_inference_bar_chart()
reporter.write_summary()
print(f"\ntotal figur: {len(list((OUT_DIR / 'figures').glob('*.png')))}")

## Ringkasan

Bahan Bab 4 lengkap: `HASIL.xlsx` di root, tabel metrik di
`outputs/tuning/metrics/`, figur di `outputs/tuning/figures/`, dan biaya
retrieval di `outputs/faiss_index/`.